In [1]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
MODEL_B = "distilbert-base-uncased"
tokenizer_b = AutoTokenizer.from_pretrained(MODEL_B)
model_b = AutoModelForQuestionAnswering.from_pretrained(MODEL_B)
MAX_LENGTH = 384
DOC_STRIDE = 96

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [2]:
def prepare_qa_features(examples):
 tokenized = tokenizer_b(
 [q.strip() for q in examples["question"]],
 examples["context"],
 truncation="only_second",
 max_length=MAX_LENGTH,
 stride=DOC_STRIDE,
 return_overflowing_tokens=True,
 return_offsets_mapping=True,
 padding="max_length",
 )
 sample_mapping = tokenized.pop("overflow_to_sample_mapping")
 offsets = tokenized.pop("offset_mapping")
 start_positions, end_positions = [], []
 for feature_index, feature_offsets in enumerate(offsets):
  input_ids = tokenized["input_ids"][feature_index]
  cls_index = input_ids.index(tokenizer_b.cls_token_id)
  sequence_ids = tokenized.sequence_ids(feature_index)
  sample_index = sample_mapping[feature_index]
  answer_start = examples["answer_start"][sample_index]
  answer_end = answer_start + len(examples["answer_text"][sample_index])
  context_start = 0
  while sequence_ids[context_start] != 1:
    context_start += 1
    context_end = len(sequence_ids) - 1
  while sequence_ids[context_end] != 1:
    context_end -= 1
  if (
  feature_offsets[context_start][0] > answer_start
  or feature_offsets[context_end][1] < answer_end
  ):
    start_positions.append(cls_index)
    end_positions.append(cls_index)
    continue
  token_start = context_start
  while feature_offsets[token_start][1] <= answer_start:
    token_start += 1
  token_end = context_end
  while feature_offsets[token_end][0] >= answer_end:
    token_end -= 1
  start_positions.append(token_start)
  end_positions.append(token_end)
 tokenized["start_positions"] = start_positions
 tokenized["end_positions"] = end_positions
 return tokenized


In [6]:
from datasets import Dataset

In [8]:
dataset = Dataset.from_json(
    "/content/technical_support_qa_100_official_docs.json"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [9]:
print(dataset)
print(dataset.column_names)
print(dataset[0])

Dataset({
    features: ['id', 'intent', 'question', 'context', 'answer_text', 'answer_start', 'source_title', 'source_url', 'source_section', 'source_type', 'context_style', 'retrieved_at', 'source_group', 'fact_group'],
    num_rows: 100
})
['id', 'intent', 'question', 'context', 'answer_text', 'answer_start', 'source_title', 'source_url', 'source_section', 'source_type', 'context_style', 'retrieved_at', 'source_group', 'fact_group']
{'id': 'QA001', 'intent': 'authentication', 'question': 'What is sufficient to use a bearer token?', 'context': 'A bearer token grants access based on possession, so it must be protected from disclosure.', 'answer_text': 'possession', 'answer_start': 38, 'source_title': 'RFC 6750 — OAuth 2.0 Bearer Token Usage', 'source_url': 'https://datatracker.ietf.org/doc/html/rfc6750', 'source_section': 'Bearer token semantics', 'source_type': 'official_documentation', 'context_style': 'faithful_paraphrase_of_source', 'retrieved_at': datetime.datetime(2026, 9, 17, 0

In [11]:
example = dataset[0]
start = example["answer_start"]
answer = example["answer_text"]

print(example["context"])
print()
print("Expected answer:", answer)
print("Extracted answer:",
      example["context"][start:start + len(answer)])

A bearer token grants access based on possession, so it must be protected from disclosure.

Expected answer: possession
Extracted answer: possession


In [12]:
split = dataset.train_test_split(
    test_size=0.30,
    seed=42
)

qa_train = split["train"]
qa_temp = split["test"]


temp_split = qa_temp.train_test_split(
    test_size=0.50,
    seed=42
)

qa_val = temp_split["train"]
qa_test = temp_split["test"]

In [14]:
qa_train_features = qa_train.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_train.column_names,
)

qa_val_features = qa_val.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_val.column_names,
)

qa_test_features = qa_test.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_test.column_names,
)

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

In [17]:
from transformers import TrainingArguments, Trainer
args_b = TrainingArguments(
 output_dir="models/qa_model",
 learning_rate=3e-5,
 per_device_train_batch_size=8,
 per_device_eval_batch_size=8,
 num_train_epochs=10,
 eval_strategy="epoch",
 save_strategy="epoch",
 load_best_model_at_end=True,
 metric_for_best_model="eval_loss",
 greater_is_better=False,
 report_to="none",
)
trainer_b = Trainer(
 model=model_b,
 args=args_b,
 train_dataset=qa_train_features,
 eval_dataset=qa_val_features,
)
trainer_b.train()
trainer_b.save_model("models/qa_model")
tokenizer_b.save_pretrained("models/qa_model")

Epoch,Training Loss,Validation Loss
1,No log,0.767615
2,No log,0.566790
3,No log,0.809176
4,No log,0.550299
5,No log,0.698070
6,No log,0.693871
7,No log,0.780687
8,No log,0.801674
9,No log,0.805992
10,No log,0.805041


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/qa_model/tokenizer_config.json', 'models/qa_model/tokenizer.json')